<a id="unit-u.828d13c39612421f838a4439524b979e"></a>

<div>
  <div>9-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
  <div>Тапсырыс фактілері, тұтынушы өлшемдері және қойма тізбегі</div>
</div>

Тұтынушы бойынша сатуға екі кесте қажет: тапсырыстар және оларды берген тұтынушылар. 7-зертханалық жұмыстың ағымдағы күйіндегі әр тапсырысқа жасалған күнімен бір жолы бар `orders_fact_l2` және әр WWI тұтынушысына бір жолы бар `customers_dim_l2` жасайсыз. Оларды біріктіріп, жолдар саны, бөлек тапсырыстар, сомалар және антибіріктіру арқылы еш тапсырыс жоғалмағанын не екі еселенбегенін дәлелдейсіз; жоспардың қалай біріктіретінін оқисыз; қате өлшем екі тексеруді де қалай бұзатынын көресіз. Содан кейін 10 және 11-жұмыстар жалғастыратын шағын жеке мысалда толық ODS → DWD → DWS → ADS тізбегін жасайсыз. Шамамен 60 минут жоспарлаңыз.

**9-модуль: деректер қоймасын модельдеу және біріктірулер** · [Сабақ](course.md) · [Тест](quiz09_modeling_and_joins.ipynb) · [Курстың README файлы](../../README.md)


<a id="unit-u.ae7160a8533d4f259a8e6a09fde8ad2e"></a>

<a id="before-you-begin"></a>

## Бастамас бұрын

Алдымен сол дерекқорда 6 және 7-зертханалық жұмыстарды аяқтаңыз. Бұл жұмыс 6-зертханалық жұмыстың `customers`, 7-зертханалық жұмыстың `orders_current` және `order_events` кестелерін оқиды, ешқайсысын өзгертпейді. Біреуі жоқ болса, баптау ұяшығы тоқтап, қай жұмысты аяқтау керегін айтады.

Жұмыс 10 және 12-жұмыстар оқитын `orders_fact_l2` және `customers_dim_l2` кестелерін қайта жасайды. 5–7-бөлімдер мысал кестелерін қайта жасайды: `ods_order_deliveries_l2`, `ods_payment_deliveries_l2`, `customers_case_dim_l2`, `dwd_orders_l2`, `dwd_payments_l2`, `dws_orders_region_day_l2` және `ads_orders_region_day_l2` көрінісі; 10-зертханалық жұмыс мысалдың DWD кестелерінен жалғасады. Бәрі зертханалық дерекқорыңызда қалады. Бәрі курс сынақ ортасында орындалады.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect
from dw_course.ui import show_sql
from dw_course.warehouse import case_fixture

lab = connect_sandbox(module=9)

<a id="unit-u.c9e400dbe1884a709b65aedc5a6eb465"></a>

<a id="1-declare-the-grain-of-the-fact-and-the-dimension"></a>

## 1. Факт пен өлшемнің түйіршіктілігін жариялау

Екі кестені біріктірмес бұрын әрқайсысының бір жолы нені білдіретінін, яғни **түйіршіктілігін** айтыңыз. `orders_fact_l2` бір жолы — қазіргі күйіндегі бір тапсырыс: 7-зертханалық жұмыстың ағымдағы күйі, сомалары және аймағы, 8-зертханалық жұмыстың көрінісіндегідей оны жасаған оқиға күнімен (`order_events` ішіндегі 1-нұсқа). Факт әр тапсырысты, бас тартылған 900002-ні де сақтайды: оны санауды есеп өз сүзгісімен шешеді. `customers_dim_l2` бір жолы — тұтынушының ағымдағы атрибуттары, мұнда 6-зертханалық жұмыстың барлық 663 WWI тұтынушысының аты.

Кесте моделі кілтті екінші рет жазудың мінез-құлқын айтады. Факт — Duplicate Key кестесі: берілген әр жолды сақтайды, бір тапсырыстың екінші жолын ештеңе бөгемейді, сондықтан ұяшық түйіршіктілікті `COUNT(*)` пен `COUNT(DISTINCT order_id)` салыстыруымен тексереді. Өлшем — `customer_id` бойынша Unique Key кестесі: тұтынушыны қайта жүктеу жолын ауыстырады, сондықтан әр тұтынушыға бір жол сақтайды.


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_fact_l2")
lab.execute("DROP TABLE IF EXISTS customers_dim_l2")
dimension_ddl = """CREATE TABLE customers_dim_l2 (
    customer_id BIGINT NOT NULL,
    customer_name VARCHAR(100) NOT NULL
)
UNIQUE KEY(customer_id)
DISTRIBUTED BY HASH(customer_id) BUCKETS 1
PROPERTIES ("replication_num"="1")"""
fact_ddl = """CREATE TABLE orders_fact_l2 (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    customer_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL,
    order_amount DECIMAL(12,2) NOT NULL,
    paid_amount DECIMAL(12,2) NOT NULL,
    refund_amount DECIMAL(12,2) NOT NULL,
    region VARCHAR(16) NOT NULL,
    data_source VARCHAR(32) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
DISTRIBUTED BY HASH(customer_id) BUCKETS 1
PROPERTIES ("replication_num"="1")"""
load_dimension = "INSERT INTO customers_dim_l2 SELECT customer_id, customer_name FROM customers"
load_fact = """INSERT INTO orders_fact_l2
SELECT DATE(e.event_time), o.order_id, o.customer_id, o.status, o.order_amount,
       o.paid_amount, o.refund_amount, o.region, o.data_source
FROM orders_current o
JOIN order_events e ON e.order_id = o.order_id AND e.event_version = 1"""
show_sql("The dimension, the fact, and their loads", ";\n\n".join([dimension_ddl, fact_ddl, load_dimension, load_fact]) + ";")
for statement in (dimension_ddl, fact_ddl, load_dimension, load_fact):
    lab.execute(statement)
lab.sql("SELECT * FROM orders_fact_l2 ORDER BY order_id", title="The order fact: one row per order")
expect(lab.query("SELECT COUNT(*), COUNT(DISTINCT order_id), SUM(order_amount) FROM orders_fact_l2"),
       [(11, 11, "1510.00")], title="Eleven rows, eleven orders, 1510.00")
expect(lab.query("SELECT COUNT(*), COUNT(DISTINCT customer_id) FROM customers_dim_l2"), [(663, 663)],
       title="663 customers, each once")

<a id="unit-u.a0803c02f583419ead65ff40316983e5"></a>

**Күтілетін нәтиже**

Екі кесте анықтамасы және жүктеулері, содан кейін он бір тапсырыс: он тапсырыстың `order_date` мәні 2026-01-01, 900011-дікі 2026-01-02; 900001 DELIVERED, 900002 CANCELLED, 900003 REFUNDED, басқалары CREATED; тұтынушылар 1–11; аймақ EAST не WEST. Екі жасыл белгі: 11 тапсырысқа 11 жол және 1510.00, әрқайсысы бір рет кездесетін 663 тұтынушы.

Факт бас тартылған тапсырыстың 200.00 сомасын қоса 1510.00 сақтайды; 8-зертханалық жұмыстың бас тартылғансыз анықтамасы 1310.00 береді. Әр есеп қайсысын қолданатынын айтса, екеуі де өз сұрағы үшін дұрыс.


<a id="unit-u.48be97d4c5ed412da2781d2e88d1b286"></a>

<a id="2-join-look-for-missing-customers-and-check-the-amounts"></a>

## 2. Біріктіру, жетіспейтін тұтынушыларды іздеу және сомаларды тексеру

`LEFT JOIN` әр тапсырысты сақтайды: тұтынушысы өлшемде жоқ тапсырыс та тұтынушы бағандары `NULL` болып көрінеді. Екінші сұрау тек сол тапсырыстарды қалдырады: `WHERE d.customer_id IS NULL` бар сол жақ біріктіру түріндегі антибіріктіру. Үшіншісі біріктіруге дейінгі және кейінгі жолдарды, бөлек тапсырыстарды және соманы салыстырады. Жетіспейтін тұтынушыларды ішкі біріктірумен тексермеңіз: ол дәл іздеген тапсырыстарыңызды алып тастайды.


In [ ]:
lab.sql("""SELECT f.order_id, f.order_date, f.status, f.order_amount, d.customer_name
FROM orders_fact_l2 f
LEFT JOIN customers_dim_l2 d ON d.customer_id = f.customer_id
ORDER BY f.order_id""", title="Each order with its customer")
lab.sql("""SELECT f.customer_id, COUNT(*) AS orders_without_customer
FROM orders_fact_l2 f
LEFT JOIN customers_dim_l2 d ON d.customer_id = f.customer_id
WHERE d.customer_id IS NULL
GROUP BY f.customer_id""", title="Orders whose customer is missing")
before_after = """SELECT 'before the join' AS stage, COUNT(*) AS row_count,
       COUNT(DISTINCT order_id) AS order_count, SUM(order_amount) AS amount
FROM orders_fact_l2
UNION ALL
SELECT 'after the left join', COUNT(*), COUNT(DISTINCT f.order_id), SUM(f.order_amount)
FROM orders_fact_l2 f LEFT JOIN customers_dim_l2 d ON d.customer_id = f.customer_id
ORDER BY stage DESC"""
lab.sql(before_after, title="Rows, orders, and amount before and after the join")
expect(lab.query(before_after), [("before the join", 11, 11, "1510.00"), ("after the left join", 11, 11, "1510.00")],
       title="The join kept every order once")

<a id="unit-u.44391058c2984ce38b182f9a1bd46158"></a>

**Күтілетін нәтиже**

Әрқайсысында WWI тұтынушы аты бар он бір тапсырыс, мысалы 1-тұтынушы үшін `Tailspin Toys (Head Office)`. Бос кесте: еш тапсырыстың тұтынушысы жетіспейді. Содан кейін біріктіруге дейін де, кейін де 11 жол, 11 тапсырыс және 1510.00, жасыл белгі.

Үш сан әртүрлі нәрсені тексереді. Тұтынушы екі рет сәйкес келсе, жолдар өсер еді; ішкі біріктіру тапсырысты алып тастаса, бөлек тапсырыстар азаяр еді; екі қатенің әсері бірін-бірі өтейтін жағдайдан басқа кезде сома екі жағдайда да өзгерер еді; келесі қадамдар әсерлердің бірін-бірі өтеуі мүмкін екенін көрсетеді.


<a id="unit-u.ffb23766a93a46c1a3c4b94756f0c333"></a>

<a id="3-aggregate-by-region-and-read-how-the-plan-joins"></a>

## 3. Аймақ бойынша агрегаттау және жоспардың біріктіруін оқу

Әр тапсырыста аймағы, яғни тіркелген сату аймағы жазылған. Біріктіргеннен кейін аймақ қорытындылары әлі фактінің 11 тапсырысы мен 1510.00 сомасын беруі керек; тексеру оларды салыстырады.

Логикалық біріктіру қай жолдар шығатынын, жоспар Doris сәйкес жолдарды қалай біріктіретінін айтады. `lab.explain_join()` жоспардың әр біріктіруін түрі мен таратылуымен көрсетеді: `BROADCAST` бір жақты, әдетте кішісін, екіншісін сканерлейтін әр данаға жібереді; `PARTITIONED` екі жақты біріктіру кілтімен қайта таратады; `BUCKET_SHUFFLE` және `COLOCATE` кестелердің бұрынғы бакеттерге бөлінуін қолданады. Курс сынақ ортасында бір BE бар, сондықтан жоспар өлшеуге емес, оқуға арналған.


In [ ]:
by_region = """SELECT f.region, COUNT(*) AS order_count, COUNT(DISTINCT d.customer_id) AS customers, SUM(f.order_amount) AS amount
FROM orders_fact_l2 f
JOIN customers_dim_l2 d ON d.customer_id = f.customer_id
GROUP BY f.region
ORDER BY f.region"""
lab.sql(by_region, title="Orders and customers per region")
expect(lab.query(by_region), [("EAST", 6, 6, "830.00"), ("WEST", 5, 5, "680.00")], title="The regions add up to the fact")
joins = lab.explain_join("""SELECT f.order_id, d.customer_name
FROM orders_fact_l2 f JOIN customers_dim_l2 d ON d.customer_id = f.customer_id""", title="How the plan joins the order fact and the customers")
expect([join["join"] for join in joins], ["INNER JOIN"], title="One inner join")

<a id="unit-u.4c46f18c37574e84a613d19b60e4278f"></a>

**Күтілетін нәтиже**

EAST — 6 тапсырыс, 6 тұтынушы, 830.00; WEST — 5 тапсырыс, 5 тұтынушы, 680.00; бірге фактінің 11 тапсырысы және 1510.00, тексеру жасыл. Содан кейін жоспар: `BROADCAST` таратылуы және `customer_id` шарты бар бір `INNER JOIN`; `orders_fact_l2` және `customers_dim_l2` екі сканерлеуі; жасыл белгі.

`BROADCAST` тұтынушыларды тапсырыстарды сканерлейтін данаға жібереді. Кластерде сол жоспар оларды тапсырыстарды сканерлейтін әр данаға көшірер еді: шағын өлшемге арзан, үлкеніне қымбат. Жоспар — Doris-тің осы деректер мен статистикаға таңдауы; Query Profile орындалудың іс жүзінде не істегенін көрсетеді.


<a id="unit-u.86357b8a7ccd44aa8b7fa089d7f049b6"></a>

<a id="4-make-a-faulty-join-visible-before-you-fix-it"></a>

## 4. Қате біріктіруді жөндемес бұрын көрінетін ету

Жоғарыдағы өлшемде әр тұтынушы бір рет бар. Екі ақаудың әсерін болжаңыз. Төмендегі сұрау тапсырыстарды CTE түріндегі қате өлшем көшірмесімен біріктіреді, сондықтан `customers_dim_l2` өзгермейді: 2-тұтынушы жоқ, 1-тұтынушы екі рет кездеседі.

- `LEFT JOIN` 2-тұтынушының тапсырысын `NULL` тұтынушы бағандарымен сақтап, 1-тұтынушы тапсырысын екі рет сәйкестендіреді.
- `INNER JOIN` 2-тұтынушы тапсырысын алып тастайды және 1-тұтынушы тапсырысын әлі екі рет сәйкестендіреді.

Біріктірілген жолдарды, бөлек тапсырыстарды, біріктірілген соманы және сәйкес келмеген тапсырыстарды салыстырыңыз. Бір жалпы сома бірін-бірі өтейтін екі қатені жасыра алады.


In [ ]:
flawed_dimension = """
WITH flawed_dim AS (
    SELECT customer_id, customer_name
    FROM customers_dim_l2 WHERE customer_id <> 2
    UNION ALL
    SELECT customer_id, customer_name
    FROM customers_dim_l2 WHERE customer_id = 1
)
"""

def join_diagnostic(join_type):
    return flawed_dimension + f"""
SELECT COUNT(*) AS joined_rows,
       COUNT(DISTINCT f.order_id) AS distinct_orders,
       SUM(f.order_amount) AS joined_amount,
       SUM(CASE WHEN d.customer_id IS NULL THEN 1 ELSE 0 END) AS unmatched_orders
FROM orders_fact_l2 AS f
{join_type} JOIN flawed_dim AS d ON d.customer_id = f.customer_id
"""

lab.sql(join_diagnostic("LEFT"), title="The faulty dimension, left join")
lab.sql(join_diagnostic("INNER"), title="The faulty dimension, inner join")
expect(lab.query(join_diagnostic("LEFT")), [(12, 11, "1610.00", 1)], title="The left join keeps the unmatched order and doubles another")
expect(lab.query(join_diagnostic("INNER")), [(11, 10, "1410.00", 0)], title="The inner join drops the unmatched order and doubles another")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_fact_l2"), [(11, "1510.00")], title="The fact is unchanged")

<a id="unit-u.62241b86b7bc4083925a8c66b6b419a2"></a>

**Күтілетін нәтиже**

Сол жақ біріктіру: 12 жол, 11 бөлек тапсырыс, 1610.00 және 1 сәйкес келмеген тапсырыс. Ішкі біріктіру: 11 жол, 10 бөлек тапсырыс, 1410.00, сәйкес келмеген тапсырыс жоқ. Үш жасыл белгі; соңғысы фактіде әлі 11 тапсырыс және 1510.00 бар дейді.

Сол жақ біріктіруде 2-тұтынушының 200.00 тапсырысы `NULL` тұтынушы бағандарымен сақталған, 1-тұтынушының 100.00 тапсырысы екі рет саналады: 12 жол, 100.00 артық. Ішкі біріктіруде 2-тұтынушы тапсырысы жоқ, 1-тұтынушыныкі әлі екі рет саналады: 11 жол, фактінің өз санындай, дұрыс көрінеді, бірақ 10 бөлек тапсырыс пен 1410.00 екі қатені де көрсетеді. Мұны еш жоспар айтпайды; тек нәтиже тексерулері айтады.


<a id="unit-u.d91c03df35c84483861357dfa34e9341"></a>

<a id="5-ods-keep-every-delivery-of-the-case"></a>

## 5. ODS: мысалдың әр жеткізілімін сақтау

1-деңгей тапсырыстары екі күнде жасалып, екі рет төленді, бұл есеп терезесіндегі төлемдерді не қайта сатып алатын тұтынушыларды өлшеуге тым аз. Сондықтан 9–11-жұмыстар жеке `datasets/warehouse_case.json` мысалын қолданады: үш күндегі 5 тұтынушының 8 тапсырысы, қайталанатын, кешігетін не қате сома әкелетін жеткізілімдер, төлемдер және ішінара қайтарым. Бұл — бөлек деректер жиыны; сомалары 1-деңгейдікі емес.

Қойма қабаттары әр кестеге бір міндет береді. ODS (Operational Data Store) қайталануларды қоса, әр жеткізілімге бір жолдан дереккөз жібергенді сақтайды; DWD (Data Warehouse Detail) бизнес түйіршіктілігінде тексерілген егжей-тегжейлі деректерді сақтайды; DWS (Data Warehouse Summary) есептер қайта қолданатын агрегаттарды сақтайды; ADS (Application Data Service) — тұтынушы оқитыны. Қабат Doris кестесінің түрі емес, модельдеу дәстүрі. 1–3-қадамдардың тапсырыс фактісі мен тұтынушы өлшемі — DWD қабатындағы 1-деңгей деректері.

Ұяшық мысал кестелерін жойып, жеткізілімдерді келген күйінде жүктейді: 10 тапсырыс жеткізілімі және 7 төлем жеткізілімі.


In [ ]:
case = case_fixture()
lab.execute("DROP VIEW IF EXISTS ads_orders_region_day_l2")
for table in ("dws_orders_region_day_l2", "dwd_orders_l2", "dwd_payments_l2", "customers_case_dim_l2", "ods_order_deliveries_l2", "ods_payment_deliveries_l2"):
    lab.execute(f"DROP TABLE IF EXISTS {table}")
lab.execute("""
CREATE TABLE ods_order_deliveries_l2 (
    input_id BIGINT NOT NULL, order_id BIGINT NOT NULL, customer_id BIGINT NOT NULL,
    created_at DATETIME NOT NULL, amount_text VARCHAR(40), status VARCHAR(20), event_version BIGINT NOT NULL
) DUPLICATE KEY(input_id) DISTRIBUTED BY HASH(input_id) BUCKETS 1 PROPERTIES ("replication_num"="1")
""")
lab.execute("""
CREATE TABLE ods_payment_deliveries_l2 (
    delivery_id BIGINT NOT NULL, payment_id VARCHAR(32) NOT NULL, order_id BIGINT NOT NULL,
    amount DECIMAL(18,2) NOT NULL, payment_time DATETIME NOT NULL
) DUPLICATE KEY(delivery_id) DISTRIBUTED BY HASH(delivery_id) BUCKETS 1 PROPERTIES ("replication_num"="1")
""")
lab.insert("ods_order_deliveries_l2", ["input_id","order_id","customer_id","created_at","amount_text","status","event_version"], case["order_deliveries"])
lab.insert("ods_payment_deliveries_l2", ["delivery_id","payment_id","order_id","amount","payment_time"], case["payment_deliveries"])
expect(lab.query("SELECT COUNT(*) FROM ods_order_deliveries_l2"), [(10,)], title="Ten order deliveries, kept as they came")
expect(lab.query("SELECT COUNT(*) FROM ods_payment_deliveries_l2"), [(7,)], title="Seven payment deliveries, the retry included")
lab.sql("SELECT input_id, order_id, amount_text, status, event_version FROM ods_order_deliveries_l2 ORDER BY input_id", title="The order deliveries in ODS");

<a id="unit-u.03720cc6ba9844088e2c29c2b4c40a8a"></a>

**Күтілетін нәтиже**

Екі жасыл белгі, 10 тапсырыс жеткізілімі және 7 төлем жеткізілімі, содан кейін тапсырыс жеткізілімдері: 2102 екі рет келеді — 2-жеткізілімде 3-нұсқа және `PARTIALLY_REFUNDED`, кейінірек 9-жеткізілімде ескі 1-нұсқа және `CREATED`; 10-жеткізілімнің 2199 тапсырысының сомасы `not-a-number`.

ODS екеуін де және қате сома мәтінін сақтайды: ережені кейін түзету оларды осы жерден қайта өңдей алады.


<a id="unit-u.5858b2629ba747238bc49368d2fe12c7"></a>

<a id="6-dwd-clean-by-business-key-not-by-delivery"></a>

## 6. DWD: жеткізілім бойынша емес, бизнес кілті бойынша тазалау

Болжаңыз: он жеткізілімнің тоғызының сомасы жарамды, оларда сегіз бөлек тапсырыс бар. 2102-нің ескі жеткізілімі оны CREATED күйіне қайтармауы керек, ал қате сома тапсырысқа айналмай ODS ішінде қалады.

Сұрау әр тапсырыс үшін ең жоғары нұсқалы бір жеткізілімді `ROW_NUMBER()` арқылы сақтайды: бұл — әр тапсырыс жолдарын (`PARTITION BY order_id`) берілген ретпен (`ORDER BY event_version DESC, input_id DESC`) нөмірлейтін терезе функциясы, сондықтан ең жаңа нұсқа 1-нөмір алады. 10-модуль терезе функцияларын түсіндіреді. Төлемдер әр `payment_id` үшін бір жол сақтайды: мысалда қайталанған төлем ID-і бірдей мазмұнда делінген, сондықтан қайта әрекет екінші төлем емес; сол ID-дің басқа сомасына жеке қайшылық ережесі қажет болар еді.


In [ ]:
lab.execute("""
CREATE TABLE customers_case_dim_l2 (customer_id BIGINT NOT NULL, region VARCHAR(16) NOT NULL)
UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1 PROPERTIES ("replication_num"="1")
""")
lab.insert("customers_case_dim_l2", ["customer_id","region"], case["customers"])
lab.execute("""
CREATE TABLE dwd_orders_l2 (
    order_id BIGINT NOT NULL, customer_id BIGINT NOT NULL, created_at DATETIME NOT NULL,
    order_amount DECIMAL(18,2) NOT NULL, status VARCHAR(20), event_version BIGINT NOT NULL
) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1", "function_column.sequence_col"="event_version")
""")
lab.execute("""
INSERT INTO dwd_orders_l2
SELECT order_id, customer_id, created_at, CAST(amount_text AS DECIMAL(18,2)), status, event_version
FROM (
    SELECT *, ROW_NUMBER() OVER(PARTITION BY order_id ORDER BY event_version DESC, input_id DESC) AS rn
    FROM ods_order_deliveries_l2
    WHERE amount_text REGEXP '^[0-9]+([.][0-9]{1,2})?$'
) valid WHERE rn=1
""")
lab.execute("""
CREATE TABLE dwd_payments_l2 (
    payment_id VARCHAR(32) NOT NULL, order_id BIGINT NOT NULL,
    amount DECIMAL(18,2) NOT NULL, payment_time DATETIME NOT NULL
) UNIQUE KEY(payment_id) DISTRIBUTED BY HASH(payment_id) BUCKETS 1 PROPERTIES ("replication_num"="1")
""")
lab.execute("""
INSERT INTO dwd_payments_l2
SELECT payment_id, order_id, amount, payment_time FROM (
    SELECT *, ROW_NUMBER() OVER(PARTITION BY payment_id ORDER BY delivery_id DESC) AS rn
    FROM ods_payment_deliveries_l2
) payments WHERE rn=1
""")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM dwd_orders_l2"), [(8, "850.00")], title="Eight current orders, 850.00, the cancelled one included")
expect(lab.query("SELECT status, event_version FROM dwd_orders_l2 WHERE order_id=2102"), [("PARTIALLY_REFUNDED", 3)], title="The late older delivery does not win")
expect(lab.query("SELECT COUNT(*) FROM dwd_payments_l2"), [(6,)], title="Six payments: the retry is not another payment")
expect(lab.query("SELECT o.order_id FROM dwd_orders_l2 o LEFT JOIN customers_case_dim_l2 c ON o.customer_id=c.customer_id WHERE c.customer_id IS NULL"), [], title="Every order has its customer")
lab.sql("SELECT order_id, customer_id, order_amount, status, event_version FROM dwd_orders_l2 ORDER BY order_id", title="The current orders in DWD");
input_summary = """SELECT COUNT(*) AS deliveries,
       SUM(CASE WHEN amount_text REGEXP '^[0-9]+([.][0-9]{1,2})?$' THEN 1 ELSE 0 END) AS valid_amount_inputs,
       SUM(CASE WHEN amount_text REGEXP '^[0-9]+([.][0-9]{1,2})?$' THEN 0 ELSE 1 END) AS rejected_amount_inputs,
       (SELECT COUNT(*) FROM dwd_orders_l2) AS current_orders
FROM ods_order_deliveries_l2"""
lab.sql(input_summary, title="From deliveries to valid inputs to current business keys")
expect(lab.query(input_summary), [(10,9,1,8)], title="Ten deliveries: nine valid inputs and one rejected; eight current orders")
lab.sql("SELECT input_id, order_id, amount_text FROM ods_order_deliveries_l2 WHERE order_id=2199 ORDER BY input_id",
        title="The rejected order is still preserved in ODS");


<a id="unit-u.a1c00b50c8634734b7f549dbe9de4b37"></a>

**Күтілетін нәтиже**

Бес жасыл белгі: 8 ағымдағы тапсырыс және 850.00, 2102 тапсырысы `PARTIALLY_REFUNDED`, 3-нұсқа, 6 төлем және әр тапсырыстың тұтынушысы бар. Содан кейін 2101–2108 сегіз тапсырыс күйі және нұсқасымен көрсетіледі; 2107 CANCELLED.

DWD тапсырыстары бас тартылған 2107 мен оның 50.00 сомасын 1-қадам фактісі 900002-ні сақтағандай сақтайды: егжей-тегжейлі деректер болғанды тіркейді, келесі қабат есеп ережесін қолданады.
Кіріс қорытындысы **10 жеткізілім = 9 жарамды кіріс + 1 жарамсыз сома**, одан кейін **8 ағымдағы тапсырыс** көрсетеді. 2102 тапсырысының екі жарамды нұсқасы бір бизнес кілтіне айналады, сондықтан сүзу мен нұсқа таңдау әртүрлі азаюды түсіндіреді. Соңғы сұрау ODS ішіндегі 2199 тапсырысының бастапқы жарамсыз сомасын көрсетеді; бастапқы жеткізілім өшірілмеген.


<a id="unit-u.88904ad2c35c4cdeb554351badc3a90e"></a>

<a id="7-dws-and-ads-aggregate-by-the-contract-publish-and-reconcile"></a>

## 7. DWS және ADS: келісімшарт бойынша агрегаттау, жариялау және салыстыру

Қорытынды тапсырыстар мен сомасын тапсырыс күні және аймақ бойынша, 8-зертханалық жұмыстың көрінісіндегідей ережемен санайды: бас тартылған тапсырыс саналмайды. Сома — төленген ақша емес, тапсырыс сомасы: 10-жұмыс төлемдерді `dwd_payments_l2` ішінен өлшейді, ішінара қайтарым тапсырыс сомасын өзгертпейді. ADS көрінісі қорытынды жолдарын бақылау тақтасы оқитын атаулармен жариялайды.

Болжаңыз: тапсырысы бар әр күн мен аймаққа біреуден бес жол, 7 тапсырыс. Ұяшық оларды мысалдың өз күтілетін жолдарымен, жалпы сомаларды DWD егжей-тегжейлі деректерімен салыстырады.


In [ ]:
lab.execute("""
CREATE TABLE dws_orders_region_day_l2 (
    order_date DATE NOT NULL, region VARCHAR(16) NOT NULL,
    order_count BIGINT NOT NULL, order_amount DECIMAL(18,2) NOT NULL
) DUPLICATE KEY(order_date, region)
DISTRIBUTED BY HASH(order_date) BUCKETS 1 PROPERTIES ("replication_num"="1")
""")
lab.execute("""
INSERT INTO dws_orders_region_day_l2
SELECT DATE(o.created_at), c.region, COUNT(*), SUM(o.order_amount)
FROM dwd_orders_l2 o JOIN customers_case_dim_l2 c ON o.customer_id=c.customer_id
WHERE o.status <> 'CANCELLED'
GROUP BY DATE(o.created_at), c.region
""")
lab.execute("""
CREATE VIEW ads_orders_region_day_l2 AS
SELECT order_date, region, order_count, order_amount FROM dws_orders_region_day_l2
""")
expect(lab.query("SELECT CAST(order_date AS STRING), region, order_count, order_amount FROM ads_orders_region_day_l2 ORDER BY order_date, region"), case["expected"]["order_daily"], title="The published rows match the case's expected rows")
expect(lab.query("SELECT SUM(order_count), SUM(order_amount) FROM ads_orders_region_day_l2"),
       lab.query("SELECT COUNT(*), SUM(order_amount) FROM dwd_orders_l2 WHERE status <> 'CANCELLED'"),
       title="ADS adds up to the DWD orders that count: 7 orders, 800.00")
lab.sql("SELECT * FROM ads_orders_region_day_l2 ORDER BY order_date, region", title="The published orders per day and region");

<a id="unit-u.9e798039a6e645a48ca67302a58d2d29"></a>

**Күтілетін нәтиже**

Екі жасыл белгі, содан кейін бес жол: 2026-01-01 EAST — 2 тапсырыс, 180.00; 2026-01-01 WEST — 1, 120.00; 2026-01-02 EAST — 1, 150.00; 2026-01-02 WEST — 1, 200.00; 2026-01-03 WEST — 2, 150.00. Барлығы 7 тапсырыс және 800.00.

Жеткізілімдер, тапсырыстар және жарияланған жолдардың сандары әртүрлі — 10, 8, 5, себебі әрқайсысының түйіршіктілігі бөлек: бір жеткізілім, бір тапсырыс, бір күн мен аймақ. Оларды сома байланыстырады: жарияланған жолдардың 800.00 сомасы — DWD тапсырыстарының 850.00 сомасынан бас тартылған 2107-нің 50.00 сомасын алып тастағандағы мән. Бұл кестелерді ODS-тен қайта құру сол жолдарды береді; қорытындыны екінші рет енгізу бермейді, себебі әр тапсырысты екі рет сақтар еді.


<a id="unit-u.d0b13ed879c84f20a29d994ff5590203"></a>

<a id="8-practice-repair-the-faulty-dimension"></a>

## 8. Жаттығу: қате өлшемді жөндеу

Фактінің әр тапсырысына дәл бір тұтынушы беретін жаңа `repaired_dim` CTE жазып, 4-қадамның төрт тексеруін сол жақ және ішкі біріктірудің екеуімен де орындаңыз. Әрқайсысынан 11 жол, 11 бөлек тапсырыс, 1510.00 және сәйкес келмеген тапсырыс жоқтығы күтіледі. `customers_dim_l2` не 1-деңгей кестелерін өзгертпеңіз: CTE тек өз сұрауыңыздағы `flawed_dim` ауыстырады.


In [ ]:
# Your code here

<a id="unit-u.bb44588653fd41ad9111639bdfcec758"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
repaired_dim = """
WITH repaired_dim AS (
    SELECT customer_id, customer_name FROM customers_dim_l2
)
"""

def repaired_join(join_type):
    return repaired_dim + f"""
SELECT COUNT(*) AS joined_rows, COUNT(DISTINCT f.order_id) AS distinct_orders,
       SUM(f.order_amount) AS joined_amount,
       SUM(CASE WHEN d.customer_id IS NULL THEN 1 ELSE 0 END) AS unmatched_orders
FROM orders_fact_l2 AS f
{join_type} JOIN repaired_dim AS d ON d.customer_id = f.customer_id
"""

lab.sql(repaired_join("LEFT"), title="The repaired dimension, left join")
expect(lab.query(repaired_join("LEFT")), [(11, 11, "1510.00", 0)], title="The left join keeps every order once")
expect(lab.query(repaired_join("INNER")), [(11, 11, "1510.00", 0)], title="The inner join keeps every order once")

<a id="unit-u.0acd7eacd9eb46da8e9a9ad0f184ca1b"></a>

**Күтілетін нәтиже**

Сол жақ біріктіру: 11 жол, 11 бөлек тапсырыс, 1510.00, сәйкес келмеген тапсырыс жоқ; сол жақ және ішкі біріктіруге екі жасыл белгі.

Қате CTE 2-тұтынушыны алып тастап, 1-тұтынушының екінші жолын қосты. Жөнделгені өлшемді бар күйінде, әр тұтынушыға бір жолмен алады; 2-тұтынушыны қайтарып, 1-тұтынушының бір жолын қалдырған CTE де сол тексерулерден өтер еді. Екі біріктіру енді бір-біріне және фактіге сәйкес, өлшемді осы фактіге жарамды ететіні сол: кілті бірегей, әр тапсырыс тұтынушысын табады.


<a id="unit-u.4ae565425b584f93b85e65a4da3467b5"></a>

<a id="you-are-done-when"></a>

## Жұмысты аяқтау шарттары

- `orders_fact_l2`, `customers_dim_l2`, `dwd_orders_l2`, `ads_orders_region_day_l2` кестелерінің бір жолы нені білдіретінін және факт неге 1510.00, есептер 1310.00 санайтынын айта аласыз;
- факт пен өлшем біріктіруі 11 тапсырыс және 1510.00 сақтайды, антибіріктіру жетіспейтін тұтынушы таппайды, жоспардан біріктіру түрі мен таратылуын оқи аласыз;
- жолдар, бөлек тапсырыстар және сомалар арқылы қате өлшем әр біріктіруге не істегенін түсіндіре аласыз;
- мысал тізбегінде 10 жеткізілім, 8 тапсырыс және 7 тапсырыс пен 800.00 бар 5 жарияланған жол сақталады, сандар неге әртүрлі екенін айта аласыз.


<a id="unit-u.266d24b889dc4c4796f01e53796c06a4"></a>

<a id="next"></a>

## Келесі қадам

[9-тест: деректер қоймасын модельдеу және біріктірулер](quiz09_modeling_and_joins.ipynb) файлын ашыңыз, содан кейін [10-модуль: метрикаларды өңдеу және ұсыну](../module10-metric-processing/course.md) бөліміне өтіңіз.
